# CE 310 — Week 11 Assignment
## Hypothesis Testing

**60 points.** Problem 1 is common to everyone; Problem 2 depends on the major
you declare below; Problem 3 is a short reflection.

Upload both CSV files to this Colab session before running anything.


## Setup


In [ ]:
# ── Identify your submission ─────────────────────────────────
# Fill these in before you run anything else. NETID is what matches your
# work to your student record — a blank NETID takes a 5-point deduction,
# and the track you do not declare in MAJOR is not graded at all.
MAJOR = ""   # "CE" for Track A, "ArcE" for Track B
NAME  = ""   # e.g. "Jordan Reyes"
NETID = ""   # e.g. "abc123"

print(f"NAME  = {NAME}")
print(f"NETID = {NETID}")
print(f"MAJOR = {MAJOR}")


Load the data and define the effect-size helper.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

items    = pd.read_csv('CE310_BidItems_2024_2026.csv')
projects = pd.read_csv('CE310_BidProjects_2024_2026.csv')

items['Ratio']    = items['UnitPrice_USD'] / items['EngineerEst_USD']
projects['Ratio'] = projects['WinningBid_USD'] / projects['EngineerEst_USD']

def cohens_d(a, b):
    """Pooled-SD effect size, matching the formula used in exercise A5."""
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp

print(f"line items {len(items):,}   awarded contracts {len(projects):,}")


## Problem 1 — Testing Extensions (30 pts)

### P1a (8 pts) — Does the Low-Bidder Result Hold on a Different Item?

Repeat exercise A2 on `MTL W-BEAM GD FEN (TIM POST)`, on the log10 scale.


In [ ]:
gf   = items[items['ItemDescription'] == 'MTL W-BEAM GD FEN (TIM POST)']
g_lo = np.log10(gf[gf['LowBidder'] == 'Yes']['UnitPrice_USD'])
g_hi = np.log10(gf[gf['LowBidder'] == 'No']['UnitPrice_USD'])

t_p1a, p_p1a = stats.ttest_ind(___, ___, equal_var=False)
d_p1a        = cohens_d(___, ___)

print(f'n = {len(gf):,}   t = {t_p1a:.4f}   p = {p_p1a:.4f}   d = {d_p1a:.4f}')

ANSWER_P1a_t = round(t_p1a, 2)
ANSWER_P1a_d = round(d_p1a, 3)
print(f'ANSWER_P1a_t = {ANSWER_P1a_t}')
print(f'ANSWER_P1a_d = {ANSWER_P1a_d}')


*Written response (4 pts) — replace this text.*

Does guard fence behave like excavation or differently? Address the colleague's
claim, and name one thing these tests cannot rule out.


### P1b (8 pts) — The Estimate Benchmark on a New Item

Run exercise A3's one-sample test on `RC PIPE (CL III)(24 IN)`.


In [ ]:
pipe = items[items['ItemDescription'] == 'RC PIPE (CL III)(24 IN)']

t_p1b, p_p1b = stats.ttest_1samp(___, 1.0)
frac_p1b     = (pipe['Ratio'] > 1.0).mean()

print(f"n = {len(pipe):,}   mean ratio = {pipe['Ratio'].mean():.4f}   median = {pipe['Ratio'].median():.4f}")
print(f't = {t_p1b:.4f}   p = {p_p1b:.4e}   share above 1.0 = {frac_p1b:.4f}')

ANSWER_P1b_t    = round(t_p1b, 2)
ANSWER_P1b_frac = round(frac_p1b, 4)
print(f'ANSWER_P1b_t = {ANSWER_P1b_t}')
print(f'ANSWER_P1b_frac = {ANSWER_P1b_frac}')


*Written response (4 pts) — replace this text.*

State both numbers and explain how a mean significantly above 1.0 coexists with
a minority of bids above the estimate. Which should the agency quote?


### P1c (7 pts, written) — Which Items Are Priced Alike?

Tukey HSD across all four line items.


In [ ]:
tukey_items = pairwise_tukeyhsd(items['Ratio'], items['ItemDescription'], alpha=0.05)
print(tukey_items)


*Written response (7 pts) — replace this text.*

How many of the six pairs are significant? Which one is not, and why might those
two items be bid alike? What is Tukey correcting for, and what breaks without it?


### P1d (7 pts) — Competition as a Grouping Variable

One-way ANOVA on the winning-bid ratio across seven bidder bands.


In [ ]:
projects['Band'] = projects['NumBidders'].clip(upper=8)

bands  = [g['Ratio'].values for _, g in projects.groupby('Band') if len(g) >= 30]
F_p1d, p_p1d = stats.f_oneway(*___)

print(projects.groupby('Band')['Ratio'].agg(['mean', 'count']).round(4))
print()
print(f'bands = {len(bands)}   F = {F_p1d:.4f}   p = {p_p1d:.4e}')

ANSWER_P1d_F = round(F_p1d, 2)
print(f'ANSWER_P1d_F = {ANSWER_P1d_F}')


*Written response (4 pts) — replace this text.*

Report F and p. Does each extra bidder help, or does the benefit arrive at once
and stop? What can ANOVA alone not tell you here?


## Problem 2 — Track Application (18 pts)

Run **only** the track matching the major you declared. Leave the other blank.

### Track A — CE: Is It Competition, or Is It Geography?


In [ ]:
METRO = ['Houston', 'Dallas', 'San Antonio', 'Austin', 'Fort Worth']
projects['Metro'] = np.where(projects['District'].isin(METRO), 'Metro', 'Non-metro')

# A2-a — winning-bid ratio, metro vs non-metro
a_m = projects[projects['Metro'] == 'Metro']['Ratio']
a_n = projects[projects['Metro'] == 'Non-metro']['Ratio']
t_a2a, p_a2a = stats.ttest_ind(___, ___, equal_var=False)
print(f'metro mean = {a_m.mean():.4f}   non-metro mean = {a_n.mean():.4f}')
print(f't = {t_a2a:.4f}   p = {p_a2a:.4e}   d = {cohens_d(a_m, a_n):.4f}')

ANSWER_A2a_t = round(t_a2a, 2)
print(f'ANSWER_A2a_t = {ANSWER_A2a_t}')


**(A2-b)** Now the number of bidders, same two groups.


In [ ]:
b_m = projects[projects['Metro'] == 'Metro']['NumBidders'].astype(float)
b_n = projects[projects['Metro'] == 'Non-metro']['NumBidders'].astype(float)
t_a2b, p_a2b = stats.ttest_ind(___, ___, equal_var=False)
print(f'metro mean bidders = {b_m.mean():.3f}   non-metro = {b_n.mean():.3f}')
print(f't = {t_a2b:.4f}   p = {p_a2b:.4e}   d = {cohens_d(b_m, b_n):.4f}')

ANSWER_A2b_t = round(t_a2b, 2)
print(f'ANSWER_A2b_t = {ANSWER_A2b_t}')


**(A2-c)** Hold competition thin — 3 bidders or fewer — and compare again.


In [ ]:
thin  = projects[projects['NumBidders'] <= 3]
c_m   = thin[thin['Metro'] == 'Metro']['Ratio']
c_n   = thin[thin['Metro'] == 'Non-metro']['Ratio']
t_a2c, p_a2c = stats.ttest_ind(___, ___, equal_var=False)
print(f'thin-competition metro n = {len(c_m)}   non-metro n = {len(c_n)}')
print(f'means {c_m.mean():.4f} vs {c_n.mean():.4f}')
print(f't = {t_a2c:.4f}   p = {p_a2c:.4f}   d = {cohens_d(c_m, c_n):.4f}')

ANSWER_A2c_t = round(t_a2c, 2)
print(f'ANSWER_A2c_t = {ANSWER_A2c_t}')


*Track A written responses (9 pts total) — replace this text.*

A2-b: what do A2-a and A2-b together suggest, and why is that not yet enough?
A2-c: what happens to the gap once competition is held thin? Which explanation
does the evidence favour, what should the agency do, and what does this design
still fail to prove?


### Track B — ArcE: Does Competition Move the Schedule Too?


In [ ]:
# B2-a — duration, Construction vs Maintenance, log10 scale
d_c = np.log10(projects[projects['ProjectType'] == 'Construction']['WorkingDays'])
d_m = np.log10(projects[projects['ProjectType'] == 'Maintenance']['WorkingDays'])
t_b2a, p_b2a = stats.ttest_ind(___, ___, equal_var=False)

print(projects.groupby('ProjectType')['WorkingDays'].median())
print(f't = {t_b2a:.4f}   p = {p_b2a:.4e}   d = {cohens_d(d_c, d_m):.4f}')

ANSWER_B2a_t = round(t_b2a, 2)
print(f'ANSWER_B2a_t = {ANSWER_B2a_t}')


**(B2-b)** How many maintenance contracts run exactly 365 days?


In [ ]:
maint = projects[projects['ProjectType'] == 'Maintenance']
frac365 = (maint['WorkingDays'] == 365).mean()
print(maint['WorkingDays'].value_counts().head(5))
print(f'share at exactly 365 days = {frac365:.4f}')

ANSWER_B2b_frac365 = round(frac365, 4)
print(f'ANSWER_B2b_frac365 = {ANSWER_B2b_frac365}')


**(B2-c)** Construction only: does duration vary with competition?


In [ ]:
con = projects[projects['ProjectType'] == 'Construction'].copy()
con['LogDays'] = np.log10(con['WorkingDays'])
con['Band']    = con['NumBidders'].clip(upper=8)

groups = [g['LogDays'].values for _, g in con.groupby('Band') if len(g) >= 30]
F_b2c, p_b2c = stats.f_oneway(*___)
print(f'bands = {len(groups)}   F = {F_b2c:.4f}   p = {p_b2c:.4f}')

few  = con[con['NumBidders'] <= 3]['LogDays']
many = con[con['NumBidders'] >= 6]['LogDays']
t_b2c, tp_b2c = stats.ttest_ind(few, many, equal_var=False)
print(f'thin vs deep: t = {t_b2c:.4f}   p = {tp_b2c:.4f}   d = {cohens_d(few, many):.4f}')

ANSWER_B2c_F = round(F_b2c, 2)
print(f'ANSWER_B2c_F = {ANSWER_B2c_F}')


*Track B written responses (9 pts total) — replace this text.*

B2-b: what does the 365-day share say about how maintenance contracts are
written, and what does it do to B2-a? B2-c: the ANOVA and the t-test disagree at
α = 0.05 — which rejects, and how is that possible on the same data? How does
the size of the competition effect on duration compare with its effect on price
in exercise B7?


## Problem 3 — Written Reflection (12 pts)

*Replace this text with your 100–150 word reflection.*

One situation where you would trust a hypothesis test to settle a construction
disagreement, and one where you would insist on seeing the distribution or a
subgroup breakdown first. Support each with a specific number from this week.


## Before You Submit

- [ ] `NAME` and `NETID` filled in at the top and showing in the cell output — a blank `NETID` takes a 5-point deduction
- [ ] `MAJOR` set to `"CE"` or `"ArcE"` — the track you do not declare is not graded
- [ ] Every cell run in order, top to bottom (Runtime → Run all) — no errors, no cell left unrun
- [ ] Every `ANSWER_` line prints a value, and no `print(f"ANSWER_... = ...")` line edited or deleted
- [ ] Every `___` replaced
- [ ] Only your declared track's cells run
- [ ] Every written response replaces its placeholder text
- [ ] Downloaded as `.ipynb` — not PDF, not `.py`
- [ ] Renamed `CE310_W11_Assignment_<NetID>.ipynb` and uploaded to the Week 11 Assignment folder on D2L